In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l5.5 Over- and underfitting

> 🎯 **Goal:** Learn to read the train-versus-validation loss curves and recognize the two classic failure modes, overfitting and underfitting, at a glance.

**Why this matters.** Now that you have a sealed validation set, the gap between train loss and validation loss becomes the single most informative picture in all of training. It tells you whether the model is genuinely learning the language or just memorizing the training text, and it tells you when to stop. Misreading these curves is how people waste days training models that were never going to generalize.

**The intuition.** Two students prepare for an exam. The first memorizes the exact answers to the practice questions: flawless on the practice set, lost on any new question. That is overfitting, training loss keeps dropping while validation loss stalls or creeps back up. The model has learned the noise and quirks of the training data instead of the underlying pattern. The second student barely studies: poor on both the practice set and the exam. That is underfitting, both losses stay stubbornly high. The model either lacks the capacity (too few parameters) or has not trained long enough to capture the pattern at all.

**The mechanics.** To force overfitting into the open, this lesson trains on a deliberately tiny 2000-character slice of the corpus. A small dataset is easy to memorize, so the symptom shows up fast. We track two curves: `tr_curve` (loss on the training slice) and `va_curve` (loss on the held-out slice), recorded every few steps with `estimate_loss`, which averages the loss over several batches for a stabler reading. Watch how they move relative to each other. Both falling together is healthy learning. The training curve diving while the validation curve flattens or rises is the unmistakable signature of overfitting.

In [ ]:
from pocketlm import (CharTokenizer, PocketLM, PocketLMConfig, init_weights,
                      make_batch, estimate_loss, split_data)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
small = torch.tensor(tok.encode(corpus[:2000]), dtype=torch.long)  # tiny on purpose
train, val = split_data(small, 0.2)
torch.manual_seed(0)
cfg = PocketLMConfig(vocab_size=tok.vocab_size, block_size=32, n_layer=2,
                     n_head=2, n_embd=64)
model = init_weights(PocketLM(cfg))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
g = torch.Generator().manual_seed(0)
steps = 80 if os.environ.get("POCKETLM_CI") else 500
tr_curve, va_curve = [], []
for s in range(steps):
    x, y = make_batch(train, 32, 16, generator=g)
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    if s % 10 == 0:
        tr_curve.append(estimate_loss(model, train, 32, 16, iters=5, generator=g))
        va_curve.append(estimate_loss(model, val, 32, 16, iters=5, generator=g))
plt.plot(tr_curve, label="train")
plt.plot(va_curve, label="val")
plt.legend()
plt.title("train vs val loss")
plt.show()
print("final train:", round(tr_curve[-1], 3), " val:", round(va_curve[-1], 3))

**What just happened.** The plot shows the two curves and the printout reports their final values. On this intentionally tiny dataset you should see the train loss far below the validation loss: the model has essentially memorized 2000 characters. That spreading gap is overfitting made visible. It is not a bug here, it is the lesson, the controlled experiment that teaches you what the symptom looks like before you meet it in the wild.

⚠️ **Common pitfalls**
- Watching only the training loss and celebrating when it drops: a falling train loss with a rising val loss is the model getting worse, not better.
- Confusing the two failure modes: overfitting is a large train-val gap; underfitting is both losses stuck high together. The fixes are opposite.
- Training past the validation minimum: once val loss stops improving, extra steps usually just deepen the overfit.

🏋️ **Try it yourself.** Replace `corpus[:2000]` with `corpus[:20000]` (ten times the data) and rerun. With more data to fit, the train-val gap should shrink noticeably, the standard real-world cure for overfitting. The four levers to remember: more data, fewer parameters, regularization (like weight decay), or early stopping when validation stops improving.

In [ ]:
assert tr_curve[-1] < tr_curve[0]   # the model did learn the train slice